# RetailOps AI
## Notebook 02 — Policy Retrieval

### Purpose
This notebook prepares the retail policy for retrieval and evaluates whether relevant policy sections can be retrieved for customer requests.

The retrieval component will later be used by the RetailOps AI agent to ground its decisions in the official retail policy.


In [5]:
from pathlib import Path
import subprocess
import os

REPO_DIR = Path("/content/tau2-bench")

if not REPO_DIR.exists():
    print("τ²-Bench not found. Cloning repository...")
    subprocess.run([
        "git",
        "clone",
        "-q",
        "https://github.com/sierra-research/tau2-bench.git",
        str(REPO_DIR)
    ], check=True)

POLICY_PATH = REPO_DIR / "data" / "tau2" / "domains" / "retail" / "policy.md"

assert POLICY_PATH.exists(), f"Policy file not found: {POLICY_PATH}"

policy_text = POLICY_PATH.read_text(encoding="utf-8")

print(" Retail policy loaded successfully.")
print("Policy path:", POLICY_PATH)
print("\nPreview:\n")
print(policy_text[:2000])

τ²-Bench not found. Cloning repository...
 Retail policy loaded successfully.
Policy path: /content/tau2-bench/data/tau2/domains/retail/policy.md

Preview:

# Retail agent policy

As a retail agent, you can help users:

- **cancel or modify pending orders**
- **return or exchange delivered orders**
- **modify their default user address**
- **provide information about their own profile, orders, and related products**

At the beginning of the conversation, you have to authenticate the user identity by locating their user id via email, or via name + zip code. This has to be done even when the user already provides the user id.

Once the user has been authenticated, you can provide the user with information about order, product, profile information, e.g. help the user look up order id.

You can only help one user per conversation (but you can handle multiple requests from the same user), and must deny any requests for tasks related to any other user.

Before taking any action that updates 

Cell 2 — Load Policy

In [6]:
from pathlib import Path

POLICY_PATH = Path("/content/tau2-bench/data/tau2/domains/retail/policy.md")

policy_text = POLICY_PATH.read_text(encoding="utf-8")

print(policy_text[:2500])

# Retail agent policy

As a retail agent, you can help users:

- **cancel or modify pending orders**
- **return or exchange delivered orders**
- **modify their default user address**
- **provide information about their own profile, orders, and related products**

At the beginning of the conversation, you have to authenticate the user identity by locating their user id via email, or via name + zip code. This has to be done even when the user already provides the user id.

Once the user has been authenticated, you can provide the user with information about order, product, profile information, e.g. help the user look up order id.

You can only help one user per conversation (but you can handle multiple requests from the same user), and must deny any requests for tasks related to any other user.

Before taking any action that updates the database (cancel, modify, return, exchange), you must list the action details and obtain explicit user confirmation (yes) to proceed.

You should not mak

Cell 3 — Split Policy into Sections

In [7]:
import re

sections = []

current_title = "General"
current_lines = []

for line in policy_text.splitlines():
    if line.startswith("#"):
        if current_lines:
            sections.append({
                "title": current_title,
                "text": "\n".join(current_lines).strip()
            })
        current_title = line.lstrip("#").strip()
        current_lines = []
    else:
        current_lines.append(line)

if current_lines:
    sections.append({
        "title": current_title,
        "text": "\n".join(current_lines).strip()
    })

sections = [s for s in sections if s["text"]]

print("Number of policy sections:", len(sections))

for s in sections[:5]:
    print("\n---", s["title"], "---")
    print(s["text"][:500])

Number of policy sections: 12

--- Retail agent policy ---
As a retail agent, you can help users:

- **cancel or modify pending orders**
- **return or exchange delivered orders**
- **modify their default user address**
- **provide information about their own profile, orders, and related products**

At the beginning of the conversation, you have to authenticate the user identity by locating their user id via email, or via name + zip code. This has to be done even when the user already provides the user id.

Once the user has been authenticated, you can p

--- Domain basic ---
- All times in the database are EST and 24 hour based. For example "02:30:00" means 2:30 AM EST.

--- User ---
Each user has a profile containing:

- unique user id
- email
- default address
- payment methods.

There are three types of payment methods: **gift card**, **paypal account**, **credit card**.

--- Product ---
Our retail store has 50 types of products.

For each **type of product**, there are **variant it

Cell 4 — Build Retrieval Index

In [8]:
!pip -q install rank-bm25

In [10]:
from rank_bm25 import BM25Okapi

def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

corpus = [
    tokenize(section["title"] + " " + section["text"])
    for section in sections
]

bm25 = BM25Okapi(corpus)

print(" BM25 policy index created.")

 BM25 policy index created.


Cell 5 — Retrieval Function

In [11]:
def retrieve_policy(query, top_k=3):
    query_tokens = tokenize(query)

    scores = bm25.get_scores(query_tokens)

    ranked = sorted(
        zip(sections, scores),
        key=lambda x: x[1],
        reverse=True
    )[:top_k]

    return [
        {
            "title": section["title"],
            "score": float(score),
            "text": section["text"]
        }
        for section, score in ranked
    ]

Cell 6 — Try Real Retail Requests

In [12]:
queries = [
    "Can I cancel my pending order?",
    "I want to return an item that was delivered.",
    "Can I change the shipping address?",
    "I want to exchange a delivered item.",
    "Can you modify my order without asking me first?"
]

for query in queries:
    print("\nQUERY:", query)

    results = retrieve_policy(query, top_k=2)

    for i, result in enumerate(results, start=1):
        print(f"\n{i}. {result['title']}")
        print("Score:", round(result["score"], 3))
        print(result["text"][:400])


QUERY: Can I cancel my pending order?

1. Cancel pending order
Score: 2.581
An order can only be cancelled if its status is 'pending', and you should check its status before taking the action.

The user needs to confirm the order id and the reason (either 'no longer needed' or 'ordered by mistake') for cancellation. Other reasons are not acceptable.

After user confirmation, the order status will be changed to 'cancelled', and the total will be refunded via the original p

2. Modify items
Score: 2.229
This action can only be called once, and will change the order status to 'pending (items modifed)'. The agent will not be able to modify or cancel the order anymore. So you must confirm all the details are correct and be cautious before taking this action. In particular, remember to remind the customer to confirm they have provided all the items they want to modify.

For a pending order, each item

QUERY: I want to return an item that was delivered.

1. Order
Score: 4.246
Each order has 

Cell 7 — Small Retrieval Evaluation

In [13]:
retrieval_tests = [
    {
        "query": "cancel pending order",
        "expected_keyword": "cancel"
    },
    {
        "query": "return delivered order",
        "expected_keyword": "return"
    },
    {
        "query": "exchange item",
        "expected_keyword": "exchange"
    },
    {
        "query": "change shipping address",
        "expected_keyword": "address"
    },
    {
        "query": "confirmation before modifying order",
        "expected_keyword": "confirm"
    }
]

correct = 0

for test in retrieval_tests:
    results = retrieve_policy(test["query"], top_k=3)

    combined_text = " ".join(
        r["title"] + " " + r["text"]
        for r in results
    ).lower()

    passed = test["expected_keyword"] in combined_text

    correct += int(passed)

    print(
        test["query"],
        "→",
        "PASS" if passed else "FAIL"
    )

retrieval_accuracy = correct / len(retrieval_tests)

print("\nRetrieval accuracy:", retrieval_accuracy)

cancel pending order → PASS
return delivered order → PASS
exchange item → PASS
change shipping address → PASS
confirmation before modifying order → PASS

Retrieval accuracy: 1.0


Cell 8 — Save Retriever Data

In [15]:
import json

output_path = Path("/content/RetailOps_AI/artifacts")

output_path.mkdir(parents=True, exist_ok=True)

with open(output_path / "policy_sections.json", "w", encoding="utf-8") as f:
    json.dump(
        sections,
        f,
        ensure_ascii=False,
        indent=2
    )

print(" Policy sections saved.")

 Policy sections saved.


## Conclusion

The retail policy was divided into structured sections and indexed using BM25 retrieval.

The retrieval component can now return relevant policy sections for common retail requests such as cancellation, returns, exchanges, address changes, and confirmation requirements.

This retriever will be integrated into the agent in the next notebook.